# Native high-level algorithm design: part I

Retain every source code stage and its register widths, including exercise/solution replays. Exercise placeholders are replaced by labeled reference solutions. Native quantum functions compose preparation, arithmetic, coherent conditions, phase and within/apply without physical wire assignments. Source struct fields become named native ports. Sampling defaults are explicit1000 local draws. Independent Circuit references compare full complex amplitudes and verify clean compiler work.


In [ ]:
import math
from typing import Annotated

from examples.synthesis.tutorials.workshops.algo_design_qce_tutorial.algo_design_qce_tutorial_part_i.overview.workflow import (
    StateCheck,
    numeric_attributes,
    phase_table,
    sample,
    verify_program,
)
from flagquantum import synth
from flagquantum.synth import (
    CX,
    H,
    Output,
    QArray,
    QBit,
    QNum,
    X,
    allocate,
    apply,
    control,
    hadamard_transform,
    phase,
    qft,
    qfunc,
    within,
)

checks: list[StateCheck] = []
actual_shots: list[int] = []


def show_stage(name: str, program: synth.Program, seed: int) -> None:
    check = verify_program(name, program)
    checks.append(check)
    print(name, "Full-state error:", check.maximum_state_error)
    print("Dirty compiler work:", check.dirty_workspace_probability)
    print(program.report["resources"])
    if name == "product_phase":
        print("Named x,y,complex amplitude:", phase_table(program))
    else:
        observations = sample(program, shots=1000, seed=seed)
        actual_shots.append(sum(row.count for row in observations))
        print("Actual1000-draw decoded observations:", observations)


@qfunc
def bell_helper(pair: QArray[QBit]) -> None:
    H(pair[0])
    CX(pair[0], pair[1])


@qfunc
def bell(res: Output[QArray[QBit]]) -> None:
    allocate(2, res)
    bell_helper(res)


## Bell preparation and actual observations

The first helper borrows the allocated pair and creates a Bell state. The named sample reader supplies decoded values rather than asking the user to decode bitstrings.


In [ ]:
bell_program = synth.synthesize(bell, memory_limit_bytes=64 * 1024**2)
show_stage("bell", bell_program, 1970)


## Exercise1 reference solution: three-qubit GHZ


In [ ]:
@qfunc
def ghz_helper(qubits: QArray[QBit]) -> None:
    H(qubits[0])
    CX(qubits[0], qubits[1])
    CX(qubits[1], qubits[2])


@qfunc
def ghz(res: Output[QArray[QBit]]) -> None:
    allocate(3, res)
    ghz_helper(res)


ghz_program = synth.synthesize(ghz, memory_limit_bytes=64 * 1024**2)
show_stage("ghz", ghz_program, 1971)


## Exercise2 reference solution: signed GHZ

Annotated QNum(3,True,0) retains signed three-bit integers. Samples report0 and-1; the raw111 word is not presented as positive7.


In [ ]:
@qfunc
def signed_ghz(x: Output[Annotated[QNum, (3, True, 0)]]) -> None:
    allocate(x)
    ghz_helper(x)


signed_program = synth.synthesize(signed_ghz, memory_limit_bytes=64 * 1024**2)
show_stage("signed_ghz", signed_program, 1972)


## Exercise3A reference solution: inferred integer addition

Inputs are two unsigned bits each. The compiler infers a four-bit output for3*a+b. Numeric attributes are read after synthesis through public program metadata. Printing or introspection inside qfunc is not required.


In [ ]:
@qfunc
def integer_sum(
    a: Output[Annotated[QNum, 2]], b: Output[Annotated[QNum, 2]], c: Output[QNum]
) -> None:
    allocate(a)
    allocate(b)
    hadamard_transform(a)
    hadamard_transform(b)
    c |= 3 * a + b


integer_program = synth.synthesize(integer_sum, memory_limit_bytes=64 * 1024**2)
print("Inferred c:", numeric_attributes(integer_program, "c"))
show_stage("integer_sum", integer_program, 1973)


## Exercise3B reference solution: binary-scaled arithmetic

a has one fractional bit,b has two. The same3*a+b expression infers an unsigned five-bit output with two fractional bits. Exact Fraction readout preserves the values.


In [ ]:
@qfunc
def fractional_sum(
    a: Output[Annotated[QNum, (2, False, 1)]],
    b: Output[Annotated[QNum, (2, False, 2)]],
    c: Output[QNum],
) -> None:
    allocate(a)
    allocate(b)
    hadamard_transform(a)
    hadamard_transform(b)
    c |= 3 * a + b


fractional_program = synth.synthesize(fractional_sum, memory_limit_bytes=64 * 1024**2)
print("Inferred c:", numeric_attributes(fractional_program, "c"))
show_stage("fractional_sum", fractional_program, 1974)


## Exercise4 reference solution: signed fractional condition

The eight x values range from-1 to3/4 in steps of1/4. Flip the coherent flag exactly when x<1/2, retaining the strict boundary and all input amplitudes.


In [ ]:
@qfunc
def signed_condition(
    x: Output[Annotated[QNum, (3, True, 2)]], flag: Output[QBit]
) -> None:
    allocate(x)
    hadamard_transform(x)
    allocate(flag)
    control(x < 0.5, lambda: X(flag))


condition_program = synth.synthesize(signed_condition, memory_limit_bytes=64 * 1024**2)
show_stage("signed_condition", condition_program, 1975)


## Exercise5 reference solution: two-round Grover

Three separate two-bit fields a/b/c replace the source struct without changing its six-bit space. Mark3*a+b+2*c==9. The zero reflection flips the joint all-zero basis state; for unsigned fields this is equivalent to a+b+c==0. Native constant phase uses phase(1,pi). The reference retains the source reflection sign, including global phase.


In [ ]:
@qfunc
def mark_equation(a: QNum, b: QNum, c: QNum) -> None:
    control(3 * a + b + 2 * c == 9, lambda: phase(1, math.pi))


@qfunc
def zero_reflection(a: QNum, b: QNum, c: QNum) -> None:
    # The three unsigned fields jointly encode zero exactly when their sum is0.
    control(a + b + c == 0, lambda: phase(1, math.pi))


@qfunc
def grover_step(a: QNum, b: QNum, c: QNum) -> None:
    mark_equation(a, b, c)
    hadamard_transform(a)
    hadamard_transform(b)
    hadamard_transform(c)
    zero_reflection(a, b, c)
    hadamard_transform(a)
    hadamard_transform(b)
    hadamard_transform(c)


@qfunc
def grover(
    a: Output[Annotated[QNum, 2]],
    b: Output[Annotated[QNum, 2]],
    c: Output[Annotated[QNum, 2]],
) -> None:
    allocate(a)
    allocate(b)
    allocate(c)
    hadamard_transform(a)
    hadamard_transform(b)
    hadamard_transform(c)
    for _ in range(2):
        grover_step(a, b, c)


grover_program = synth.synthesize(grover, memory_limit_bytes=64 * 1024**2)
show_stage("grover", grover_program, 1976)


## Exercise6A reference solution: product phase

Encode x*y with coefficient pi/8 for two two-bit registers. Probabilities alone hide this effect; the table exposes actual complex amplitudes after verifying zero compiler work. No phase alignment is used in the independent check.


In [ ]:
@qfunc
def product_phase(x: Output[Annotated[QNum, 2]], y: Output[Annotated[QNum, 2]]) -> None:
    allocate(x)
    allocate(y)
    hadamard_transform(x)
    hadamard_transform(y)
    phase(x * y, math.pi / 8)


phase_program = synth.synthesize(product_phase, memory_limit_bytes=64 * 1024**2)
show_stage("product_phase", phase_program, 1977)


## Exercise6B reference solution: Fourier-basis square

Keep three-bit x and four-bit y. within/apply automatically reverses QFT after the phase expression, realizing y+=x*x modulo16. The Circuit reference uses an independently enumerated addition permutation, rather than the Fourier implementation.


In [ ]:
@qfunc
def fourier_square(
    x: Output[Annotated[QNum, 3]], y: Output[Annotated[QNum, 4]]
) -> None:
    allocate(x)
    hadamard_transform(x)
    allocate(y)
    with within():
        qft(y)
    with apply():
        phase(y * (x**2), 2 * math.pi / 16)


fourier_program = synth.synthesize(fourier_square, memory_limit_bytes=64 * 1024**2)
show_stage("fourier_square", fourier_program, 1978)


## Source solution replay: ghz

Rebuild and execute the same native solution, as the original solution section does. Keep independent state verification and actual observations; do not replace them with known solutions.


In [ ]:
ghz_solution = synth.synthesize(ghz, memory_limit_bytes=64 * 1024**2)
show_stage("ghz", ghz_solution, 1980)


## Source solution replay: signed ghz

Rebuild and execute the same native solution, as the original solution section does. Keep independent state verification and actual observations; do not replace them with known solutions.


In [ ]:
signed_ghz_solution = synth.synthesize(signed_ghz, memory_limit_bytes=64 * 1024**2)
show_stage("signed_ghz", signed_ghz_solution, 1981)


## Source solution replay: integer sum

Rebuild and execute the same native solution, as the original solution section does. Keep independent state verification and actual observations; do not replace them with known solutions.


In [ ]:
integer_sum_solution = synth.synthesize(integer_sum, memory_limit_bytes=64 * 1024**2)
print("Inferred c:", numeric_attributes(integer_sum_solution, "c"))
show_stage("integer_sum", integer_sum_solution, 1982)


## Source solution replay: fractional sum

Rebuild and execute the same native solution, as the original solution section does. Keep independent state verification and actual observations; do not replace them with known solutions.


In [ ]:
fractional_sum_solution = synth.synthesize(
    fractional_sum, memory_limit_bytes=64 * 1024**2
)
print("Inferred c:", numeric_attributes(fractional_sum_solution, "c"))
show_stage("fractional_sum", fractional_sum_solution, 1983)


## Source solution replay: signed condition

Rebuild and execute the same native solution, as the original solution section does. Keep independent state verification and actual observations; do not replace them with known solutions.


In [ ]:
signed_condition_solution = synth.synthesize(
    signed_condition, memory_limit_bytes=64 * 1024**2
)
show_stage("signed_condition", signed_condition_solution, 1984)


## Source solution replay: grover

Rebuild and execute the same native solution, as the original solution section does. Keep independent state verification and actual observations; do not replace them with known solutions.


In [ ]:
grover_solution = synth.synthesize(grover, memory_limit_bytes=64 * 1024**2)
show_stage("grover", grover_solution, 1985)


## Source solution replay: product phase

Rebuild and execute the same native solution, as the original solution section does. Keep independent state verification and actual observations; do not replace them with known solutions.


In [ ]:
product_phase_solution = synth.synthesize(
    product_phase, memory_limit_bytes=64 * 1024**2
)
show_stage("product_phase", product_phase_solution, 1986)


## Source solution replay: fourier square

Rebuild and execute the same native solution, as the original solution section does. Keep independent state verification and actual observations; do not replace them with known solutions.


In [ ]:
fourier_square_solution = synth.synthesize(
    fourier_square, memory_limit_bytes=64 * 1024**2
)
show_stage("fourier_square", fourier_square_solution, 1987)


All stages are local CPU work. Resource counts are synthesized instructions before Runtime compilation; they are not hardware-basis counts. The original sizes and both exercise/solution execution sections are retained. No hardware submission,scalability,speedup or optimality claim is made.
